# Torsion Scan Guardian — multi-molecule sweep on Colab

Runs the full active-learning pipeline on every molecule in
[`data/molecule_library/candidates.csv`](../data/molecule_library/candidates.csv)
(or any subset you pick), in series, on Google Colab's free T4 GPU.

Per molecule: build seed dataset → fine-tune 3 ensemble members → calibrate threshold → run **baseline** MD (Guardian off) → run **AL** MD (cloud-5, safeguarded fine-tune) → compute stability metrics. Per-molecule wall time: **~10–20 min on GPU.**

**Sweep cost estimates** (4000 steps per run, T4 GPU):
- 3 `todo` molecules (glycine zwitterion, sulfonyl chloride, diglycine): **~45–60 min**
- 4 `candidate` molecules added: **~+50 min** (90 min total)
- All 9 (including the 2 already-done): **~2–3 hours**

Colab's free tier has a ~90-min idle timeout. The sweep writes progress to `runs/sweep/sweep_summary.csv` **after every molecule** so a disconnect doesn't lose finished work — just re-run the sweep cell and finished molecules' seed/fine-tune artifacts are reused.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## 1. GPU + repo + dependencies

If you've already run [`guardian_colab.ipynb`](guardian_colab.ipynb) in this Drive, cells 1–4 here are essentially no-ops (env is cached, repo is cloned). Otherwise it's the same setup.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
REPO_URL = 'https://github.com/Sathapana/torsion_scan_guardian.git'
REPO_DIR = '/content/drive/MyDrive/torsion-scan-guardian'
!test -d torsion-scan-guardian || git clone $REPO_URL torsion-scan-guardian
%cd torsion-scan-guardian
# Pull latest in case you ran sessions earlier and pushed updates since.
!git pull --ff-only origin main

In [ ]:
import os
assert os.path.exists(os.path.join(REPO_DIR, 'scripts/sweep_molecules.py')), \
    f'sweep script missing at {REPO_DIR}/scripts/sweep_molecules.py'
print('OK — sweep script present')

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q mace-torch ase rdkit matplotlib pandas pydantic pyyaml tqdm pytest numpy
cd /content/drive/MyDrive/torsion-scan-guardian
pip install -q -e .

In [ ]:
# xtb (for GFN-FF oracle) via condacolab — RESTARTS THE KERNEL.
# After kernel restart, re-run cells 1, 2, 3, 4 above (verify GPU, mount Drive,
# sanity check, pip install), then continue from cell 6 below.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# After kernel restart, install xtb itself.
!conda install -y -c conda-forge xtb 2>&1 | tail -3
!which xtb && xtb --version 2>&1 | head -3

In [ ]:
# Pre-download MACE-OFF small (~7 MB) so the sweep's fine-tune subprocess
# doesn't fail per-molecule on a missing cache. Idempotent: re-runs are no-ops.
import torch
from mace.calculators import mace_off
calc = mace_off(model='small', device='cuda' if torch.cuda.is_available() else 'cpu')
import os
cache_path = os.path.expanduser('~/.cache/mace/MACE-OFF23_small.model')
assert os.path.exists(cache_path), f'MACE-OFF cache file missing at {cache_path} after download'
print(f'MACE-OFF cache ready: {cache_path}  ({os.path.getsize(cache_path) // 1024} KB)')

## 2. Inspect the molecule catalog

Edit `data/molecule_library/candidates.csv` directly if you want to add or change a row.

In [ ]:
import pandas as pd
cat = pd.read_csv('data/molecule_library/candidates.csv')
cat[['name', 'smiles', 'charge', 'heavy_atoms', 'phase', 'why_interesting']]

## 3. Pick the molecules to sweep

Two ways to choose:
- **By name** (most explicit): set `MOLECULES` to a list of `name` values from the catalog.
- **By phase**: leave `MOLECULES = None` and set `PHASE_FILTER` to one or more of `todo`, `candidate`, `done`.

Default below is the 3 `todo` molecules — the natural Phase-6 sweep.

In [ ]:
# === EDIT HERE ===
MOLECULES = None                          # e.g. ['glycine_zwitterion', 'biphenyl']; None = use PHASE_FILTER
PHASE_FILTER = ['todo']                   # one or more of 'todo', 'candidate', 'done'
STEPS = 4000                              # MD step budget per run (baseline + AL each)
TEMPERATURE = 300                         # K
CLOUD_SIZE = 5                            # number of perturbed labels per trigger
MAX_TRIGGERS = 5                          # max AL cycles per molecule
BASELINE_ONLY = False                     # set True for a first-pass screen with no AL
THRESHOLD_OVERRIDE = None                 # e.g. 0.05; None = per-molecule calibration
# =================

# Resolve what will actually run.
if MOLECULES:
    selected = cat[cat['name'].isin(MOLECULES)]
else:
    selected = cat[cat['phase'].isin(PHASE_FILTER)]
print(f'Will sweep {len(selected)} molecule(s):')
selected[['name', 'smiles', 'heavy_atoms', 'phase']]

## 4. Run the sweep

Streams per-molecule progress. Each molecule's results are persisted to `runs/sweep/sweep_summary.csv` immediately, so a Colab disconnect can be recovered just by re-running this cell.

In [ ]:
import subprocess, sys, os

cmd = [sys.executable, 'scripts/sweep_molecules.py',
       '--steps', str(STEPS),
       '--temperature', str(TEMPERATURE),
       '--cloud-size', str(CLOUD_SIZE),
       '--max-triggers', str(MAX_TRIGGERS),
       '--device', 'cuda']
if MOLECULES:
    cmd += ['--molecules', *MOLECULES]
else:
    cmd += ['--phase-filter', *PHASE_FILTER]
if BASELINE_ONLY:
    cmd.append('--baseline-only')
if THRESHOLD_OVERRIDE is not None:
    cmd += ['--threshold', str(THRESHOLD_OVERRIDE)]

env = dict(os.environ); env['PYTHONIOENCODING'] = 'utf-8'; env['MPLBACKEND'] = 'Agg'

# Stream stdout line-by-line so progress is visible.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding='utf-8', env=env, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\nsweep exit code: {rc}')

## 5. Review the aggregate summary

In [ ]:
summary = pd.read_csv('runs/sweep/sweep_summary.csv')
# Only show the columns most useful to scan first; everything else is in the CSV.
cols = ['name', 'status', 'heavy_atoms', 'calibrated_threshold',
        'baseline_wall_s', 'baseline_max_bond_stretch', 'baseline_broken_bonds_final',
        'al_wall_s', 'al_triggers', 'al_labels',
        'al_max_bond_stretch', 'al_broken_bonds_final', 'total_wall_s']
available_cols = [c for c in cols if c in summary.columns]
summary[available_cols]

## 6. Plot the comparison

Two diagnostic plots:
- **Max bond stretch (AL vs baseline)** — if AL is stabilising a divergent baseline, AL bars should be much shorter (closer to 1.0); baseline bars near or above 1.6 mean the molecule collapsed without help.
- **Triggers fired** — a non-zero count means the Guardian detected something. The higher the count, the more cycles the AL machinery actually exercised.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = summary[summary['status'] == 'ok'].copy()
names = df['name'].tolist()
x = np.arange(len(names))
w = 0.4

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(x - w/2, df['baseline_max_bond_stretch'], w, label='baseline', color='#7f7f7f')
if 'al_max_bond_stretch' in df.columns:
    axes[0].bar(x + w/2, df['al_max_bond_stretch'], w, label='AL (Guardian)', color='#1f77b4')
axes[0].axhline(1.6, color='#d62728', linestyle='--', lw=0.7, label='bond-break threshold (1.6x)')
axes[0].axhline(1.0, color='black', lw=0.5)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].set_ylabel('max bond stretch ratio')
axes[0].set_title('Stability per molecule (lower = better)')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

if 'al_triggers' in df.columns:
    axes[1].bar(x, df['al_triggers'].fillna(0), color='#2ca02c')
    axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=30, ha='right')
    axes[1].set_ylabel('AL cycles completed')
    axes[1].set_title('Active-learning triggers fired')
    axes[1].grid(axis='y', alpha=0.3)
else:
    axes[1].set_visible(False)

fig.suptitle('Multi-molecule sweep — AL vs baseline', y=1.02)
fig.tight_layout()
fig.savefig('figures/sweep_summary.png', dpi=140, bbox_inches='tight')
plt.show()
print('Wrote figures/sweep_summary.png')

## 7. Bundle results for download

Same pattern as [`guardian_colab.ipynb`](guardian_colab.ipynb) section 9 — zips the small version-worthy artifacts (the sweep summary CSV, per-molecule figures, the new sweep_summary plot, summary.json from every run) and triggers a browser download.

Unzip into the repo on your local machine, review with `git status`, then `git add` + `git commit` + `git push` from there. Keeps GitHub credentials out of Colab.

In [ ]:
import shutil, os, glob
from google.colab import files

bundle_dir = '/content/guardian_sweep_results'
os.makedirs(bundle_dir, exist_ok=True)

def cp(src, dst):
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy2(src, dst)

# Sweep-level artifacts.
for p in ['runs/sweep/sweep_summary.csv', 'figures/sweep_summary.png']:
    if os.path.exists(p):
        cp(p, bundle_dir + '/' + os.path.basename(p))

# Per-molecule summary.json files — flatten into bundle/<molecule>_<mode>_summary.json
for s in glob.glob('runs/sweep/*/*/summary.json'):
    parts = s.replace('\\', '/').split('/')
    mol, mode = parts[-3], parts[-2]
    cp(s, f'{bundle_dir}/{mol}_{mode}_summary.json')

# Per-molecule figures the analysis script produced.
for p in glob.glob('figures/*.png'):
    cp(p, bundle_dir + '/' + os.path.basename(p))

# Seed datasets for newly-processed molecules (useful for reproducibility).
for p in glob.glob('data/seed/*.xyz'):
    cp(p, bundle_dir + '/' + os.path.basename(p))

# Auto-created per-molecule configs.
if os.path.isdir('config/molecules'):
    cp('config/molecules', bundle_dir + '/molecules')

print('Files in bundle:')
for f in sorted(os.listdir(bundle_dir)):
    print('  ' + f)

shutil.make_archive('/content/guardian_sweep_results', 'zip', bundle_dir)
files.download('/content/guardian_sweep_results.zip')